# Armed Corridor Diagnostic Runner

Focused Colab runner for the armed-corridor state -> visible -> hidden progression. This is separate from `colab_runner.ipynb` so the generic Stage 0 suite stays untouched.

Default suite:
- benchmarks: `armed_corridor_state`, `armed_corridor_visible`, `armed_corridor`
- methods: `no_concept`, `vanilla_freeze`, `concept_actor_critic`
- temporal encoding: `gru`
- budgets: 100k for state, 300k for visible/hidden

Use this to isolate whether failures are from reward/control dynamics, pixel representation, hidden temporal state, or concept bottleneck behavior.

In [ ]:
# 1) Mount Drive + configure armed-corridor-only output
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/concept_critic'
except ImportError:
    print('Not running on Colab - using local /tmp for outputs')
    DRIVE_ROOT = '/tmp/concept_critic'

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)

OUTPUT_DIR = f'{DRIVE_ROOT}/armed_corridor_diagnostics'
MAX_MINUTES = 660
os.environ['DRIVE_ROOT'] = DRIVE_ROOT
os.environ['OUTPUT_DIR'] = OUTPUT_DIR
os.environ['MAX_MINUTES'] = str(MAX_MINUTES)

print('OUTPUT_DIR:', OUTPUT_DIR)
print('MAX_MINUTES:', MAX_MINUTES)

In [ ]:
# 2) Locate or fetch repo + verify armed runner support
import os, subprocess, sys

REPO_URL = 'https://github.com/AdeX11/concept_critic_models.git'
BRANCH = 'domingo-experimental'

def _find_repo_root() -> str | None:
    cur = os.path.abspath(os.getcwd())
    for _ in range(6):
        if os.path.exists(os.path.join(cur, 'train.py')):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    if os.path.exists('/content/repo/train.py'):
        return '/content/repo'
    return None

REPO_DIR = _find_repo_root()
if REPO_DIR is None:
    if not os.path.isdir('/content'):
        raise RuntimeError('Open this notebook from inside the cloned repo, or run it on Colab.')
    REPO_DIR = '/content/repo'
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)
print('repo  :', REPO_DIR)
print('head  :', subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip())
print('branch:', subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD']).decode().strip())

runner = os.path.join(REPO_DIR, 'colab', 'run_armed_corridor_suite.py')
if not os.path.exists(runner):
    raise RuntimeError('Missing colab/run_armed_corridor_suite.py. Pull the latest domingo-experimental branch.')
subprocess.check_call([sys.executable, '-m', 'py_compile', runner])
RUN_SUITE_LIVE_FLAG = '--stream_train_logs'
os.environ['RUN_SUITE_LIVE_FLAG'] = RUN_SUITE_LIVE_FLAG
print('armed corridor runner: ready')

In [ ]:
# 3) Install dependencies + smoke checks
!bash colab/setup.sh

## Run Armed Corridor Diagnostic Suite

This runs 9 default jobs: 3 methods x 3 armed-corridor variants. Completed runs are skipped by `eval.json`.

In [ ]:
!python colab/run_armed_corridor_suite.py \
    --output_dir "$OUTPUT_DIR" \
    --max_minutes "$MAX_MINUTES" \
    $RUN_SUITE_LIVE_FLAG

## Optional CAC None Ablation

Run this after the default suite only if time remains. It adds `concept_actor_critic + none` for each armed-corridor variant.

In [ ]:
!python colab/run_armed_corridor_suite.py \
    --methods concept_actor_critic \
    --temporal_encodings gru \
    --include_cac_none \
    --output_dir "$OUTPUT_DIR" \
    --max_minutes "$MAX_MINUTES" \
    $RUN_SUITE_LIVE_FLAG

## Aggregate + Inspect

In [ ]:
!python cluster/aggregate.py "$OUTPUT_DIR" --csv-out "$OUTPUT_DIR/armed_corridor_summary.csv"
import pandas as pd
df = pd.read_csv(f'{OUTPUT_DIR}/armed_corridor_summary.csv')
print('total runs:', len(df))
print(df.groupby(['benchmark_id', 'method', 'temporal_encoding']).size().rename('count'))
cols = ['benchmark_id','method','temporal_encoding','mean_reward','success_rate','dominant_action_fraction','terminal_cause_breakdown','concept_metrics','concept_diagnostics']
print(df[[c for c in cols if c in df.columns]].to_string(index=False))

In [ ]:
# Inspect one run's live/persisted training log and TensorBoard scalar summary
RUN = 'concept_actor_critic_two_phase_gru_armed_corridor_seed42'
!tail -80 "{OUTPUT_DIR}/{RUN}/train.log"
!python colab/summarize_tensorboard.py "{OUTPUT_DIR}/{RUN}"